In [1]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!rm -f spark-3.5.1-bin-hadoop3.tgz
!wget https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

--2026-04-14 18:29:09--  https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400446614 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.1-bin-hadoop3.tgz’

spark-3.5.1-bin-had 100%[===================>] 381.90M  2.11MB/s    in 7m 48s  

2026-04-14 18:36:58 (835 KB/s) - ‘spark-3.5.1-bin-hadoop3.tgz’ saved [400446614/400446614]



In [24]:
#importação
import os
import findspark
import glob
import sys

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"


os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print(f"PYSPARK_PYTHON set to: {os.environ['PYSPARK_PYTHON']}")

pyspark_zip_path = glob.glob(os.path.join(os.environ["SPARK_HOME"], "python/lib/pyspark*.zip"))
py4j_src_zip_path = glob.glob(os.path.join(os.environ["SPARK_HOME"], "python/lib/py4j-*-src.zip"))

python_paths_to_add = []
if pyspark_zip_path:
    python_paths_to_add.append(pyspark_zip_path[0])
    print(f"Added pyspark.zip to PYTHONPATH: {pyspark_zip_path[0]}")
if py4j_src_zip_path:
    python_paths_to_add.append(py4j_src_zip_path[0])
    print(f"Added py4j-src.zip to PYTHONPATH: {py4j_src_zip_path[0]}")

current_pythonpath_items = [p for p in os.environ.get("PYTHONPATH", "").split(':') if p]
final_pythonpath_list = python_paths_to_add + [item for item in current_pythonpath_items if item not in python_paths_to_add]
os.environ["PYTHONPATH"] = ":".join(final_pythonpath_list)
print(f"Final PYTHONPATH: {os.environ['PYTHONPATH']}")


spark_home = os.environ["SPARK_HOME"]
all_jars = glob.glob(os.path.join(spark_home, "jars", "*.jar"))

full_classpath_str = ":".join(all_jars)
print(f"Constructed full_classpath_str for PYSPARK_SUBMIT_ARGS: {full_classpath_str[:200]}...")

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--driver-class-path {full_classpath_str} pyspark-shell"
print(f"PYSPARK_SUBMIT_ARGS set to: {os.environ['PYSPARK_SUBMIT_ARGS'][:200]}...")


spark_home_path = os.environ["SPARK_HOME"]
if not os.path.exists(spark_home_path) or not os.path.isdir(spark_home_path):
    print(f"Erro: O diretório SPARK_HOME '{spark_home_path}' não foi encontrado ou não é um diretório válido.")
    print("Por favor, verifique se a instalação do Apache Spark no(s) passo(s) anterior(es) foi bem-sucedida.")
else:
    try:
        findspark.init(spark_home=spark_home_path)
        print("findspark inicializado com sucesso!")
    except Exception as e:
        print(f"Erro ao inicializar findspark: {e}")
        print("Verifique se o Apache Spark foi extraído corretamente no diretório SPARK_HOME.")

PYSPARK_PYTHON set to: /usr/bin/python3
Added pyspark.zip to PYTHONPATH: /content/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip
Added py4j-src.zip to PYTHONPATH: /content/spark-3.5.1-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip
Final PYTHONPATH: /content/spark-3.5.1-bin-hadoop3/python/lib/pyspark.zip:/content/spark-3.5.1-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip:/env/python
Constructed full_classpath_str for PYSPARK_SUBMIT_ARGS: /content/spark-3.5.1-bin-hadoop3/jars/hive-shims-scheduler-2.3.9.jar:/content/spark-3.5.1-bin-hadoop3/jars/metrics-core-4.2.19.jar:/content/spark-3.5.1-bin-hadoop3/jars/commons-crypto-1.1.0.jar:/conte...
PYSPARK_SUBMIT_ARGS set to: --driver-class-path /content/spark-3.5.1-bin-hadoop3/jars/hive-shims-scheduler-2.3.9.jar:/content/spark-3.5.1-bin-hadoop3/jars/metrics-core-4.2.19.jar:/content/spark-3.5.1-bin-hadoop3/jars/commons-cry...
findspark inicializado com sucesso!


In [3]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

if 'spark' in globals() and spark.sparkContext._jsc is not None:
    print("Stopping existing SparkSession...")
    spark.stop()
    print("Existing SparkSession stopped.")

spark = SparkSession.builder \
    .appName("Agregacao") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "10g") \
    .config("spark.sql.warehouse.dir", "file:///tmp/spark-warehouse") \
    .getOrCreate()

print("Spark iniciado com sucesso!")

Spark iniciado com sucesso!


In [4]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [5]:
!ls -l /content

total 391320
drwxr-xr-x  1 root root      4096 Apr  2 13:31 sample_data
drwxr-xr-x 13 1000 1000      4096 Feb 15  2024 spark-3.5.1-bin-hadoop3
-rw-r--r--  1 root root 400446614 Feb 15  2024 spark-3.5.1-bin-hadoop3.tgz
-rw-r--r--  1 root root    245808 Apr 14 17:34 videos-preparados.snappy.parquet


In [6]:
from google.colab import files

uploaded = files.upload()

Saving videos-preparados.snappy.parquet to videos-preparados.snappy (1).parquet


In [7]:
df_video = spark.read.parquet("videos-preparados.snappy.parquet")

df_video.printSchema()
df_video.show(5, truncate=False)

root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)

+-----------------------------------------------------------------------------------------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+---------------------+----------------------------------------------------------------+--------------------------------------+
|Title                                                                                             

In [8]:
!pip install pyspark

In [ ]:
print('Listando o conteúdo do diretório SPARK_HOME:')
!ls -R /content/spark-3.5.1-bin-hadoop3

Listando o conteúdo do diretório SPARK_HOME:
/content/spark-3.5.1-bin-hadoop3:
bin   data	jars	    LICENSE   NOTICE  R		 RELEASE  yarn
conf  examples	kubernetes  licenses  python  README.md  sbin

/content/spark-3.5.1-bin-hadoop3/bin:
beeline		      pyspark		spark-class.cmd      spark-shell.cmd
beeline.cmd	      pyspark2.cmd	spark-connect-shell  spark-sql
docker-image-tool.sh  pyspark.cmd	sparkR		     spark-sql2.cmd
find-spark-home       run-example	sparkR2.cmd	     spark-sql.cmd
find-spark-home.cmd   run-example.cmd	sparkR.cmd	     spark-submit
load-spark-env.cmd    spark-class	spark-shell	     spark-submit2.cmd
load-spark-env.sh     spark-class2.cmd	spark-shell2.cmd     spark-submit.cmd

/content/spark-3.5.1-bin-hadoop3/conf:
fairscheduler.xml.template  metrics.properties.template   spark-env.sh.template
log4j2.properties.template  spark-defaults.conf.template  workers.template

/content/spark-3.5.1-bin-hadoop3/data:
artifact-tests	graphx	mllib  streaming

/content/spark-3.5.1-bin-ha

In [9]:
df_video.groupBy("Keyword") \
    .count() \
    .orderBy("Keyword") \
    .show(truncate=False)

+----------------+-----+
|Keyword         |count|
+----------------+-----+
|animals         |38   |
|apple           |42   |
|asmr            |49   |
|bed             |44   |
|biology         |47   |
|business        |48   |
|chess           |47   |
|cnn             |50   |
|computer science|48   |
|crypto          |50   |
|cubes           |49   |
|data science    |50   |
|education       |24   |
|finance         |39   |
|food            |48   |
|game development|50   |
|gaming          |42   |
|google          |45   |
|history         |49   |
|how-to          |48   |
+----------------+-----+
only showing top 20 rows



In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

if 'spark' in globals() and spark.sparkContext._jsc is not None:
    print("Stopping existing SparkSession...")
    spark.stop()
    print("Existing SparkSession stopped.")

sprk = SparkSession.builder \
    .appName("Agregacao") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "10g") \
    .config("spark.sql.warehouse.dir", "file:///tmp/spark-warehouse") \
    .getOrCreate()

print("Spark iniciado com sucesso!")

Stopping existing SparkSession...
Existing SparkSession stopped.
Spark iniciado com sucesso!


In [12]:
df_video = sprk.read.parquet("videos-preparados.snappy.parquet")
df_video.groupBy("Keyword") \
    .agg(F.avg("Interaction").alias("Media_Interaction")) \
    .orderBy("Keyword") \
    .show(truncate=False)

+----------------+---------------------+
|Keyword         |Media_Interaction    |
+----------------+---------------------+
|animals         |9.55066085263158E7   |
|apple           |1.0873628214285715E7 |
|asmr            |1779749.693877551    |
|bed             |5.438209175E7        |
|biology         |4192382.063829787    |
|business        |7310180.020833333    |
|chess           |1360808.6595744682   |
|cnn             |570650.86            |
|computer science|1226793.0208333333   |
|crypto          |413676.2             |
|cubes           |1.5043961224489795E7 |
|data science    |562465.28            |
|education       |2750838.625          |
|finance         |708542.9487179487    |
|food            |5352944.104166667    |
|game development|752243.56            |
|gaming          |520080.38095238095   |
|google          |-3.3510236088888887E7|
|history         |1.565269257142857E7  |
|how-to          |7975134.5            |
+----------------+---------------------+
only showing top

In [13]:
df_video.groupBy("Keyword") \
    .agg(
        F.avg("Views").alias("Media_Views"),
        F.variance("Views").alias("Variancia_Views")
    ) \
    .orderBy("Keyword") \
    .show(truncate=False)

+----------------+--------------------+----------------------+
|Keyword         |Media_Views         |Variancia_Views       |
+----------------+--------------------+----------------------+
|animals         |9.472396092105263E7 |8.3537868257472992E16 |
|apple           |1.0746930452380951E7|4.299927977442589E15  |
|asmr            |1741148.0408163266  |3.4972325861033168E13 |
|bed             |5.389322861363637E7 |1.1661048318545686E16 |
|biology         |4121605.829787234   |2.589074643476953E13  |
|business        |7236354.520833333   |9.546545645303889E14  |
|chess           |1323293.9574468085  |6.391876442014173E12  |
|cnn             |554240.38           |1.5634236184681186E11 |
|computer science|1191958.7083333333  |2.81219868165102E12   |
|crypto          |404608.22           |3.513691634369074E12  |
|cubes           |1.4735344122448979E7|8.511756571903261E14  |
|data science    |544771.98           |5.4793365253499945E11 |
|education       |2684432.8333333335  |1.83357224933921

In [14]:
df_video.groupBy("Keyword") \
    .agg(
        F.avg("Views").alias("Media_Views"),
        F.variance("Views").alias("Variancia_Views")
    ) \
    .orderBy("Keyword") \
    .show(truncate=False)


+----------------+--------------------+----------------------+
|Keyword         |Media_Views         |Variancia_Views       |
+----------------+--------------------+----------------------+
|animals         |9.472396092105263E7 |8.3537868257472992E16 |
|apple           |1.0746930452380951E7|4.299927977442589E15  |
|asmr            |1741148.0408163266  |3.4972325861033168E13 |
|bed             |5.389322861363637E7 |1.1661048318545686E16 |
|biology         |4121605.829787234   |2.589074643476953E13  |
|business        |7236354.520833333   |9.546545645303889E14  |
|chess           |1323293.9574468085  |6.391876442014173E12  |
|cnn             |554240.38           |1.5634236184681186E11 |
|computer science|1191958.7083333333  |2.81219868165102E12   |
|crypto          |404608.22           |3.513691634369074E12  |
|cubes           |1.4735344122448979E7|8.511756571903261E14  |
|data science    |544771.98           |5.4793365253499945E11 |
|education       |2684432.8333333335  |1.83357224933921

In [15]:
df_video.groupBy("Keyword") \
    .agg(
        F.round(F.avg("Views"), 0).cast("long").alias("Media_Views"),
        F.round(F.min("Views"), 0).cast("long").alias("Min_Views"),
        F.round(F.max("Views"), 0).cast("long").alias("Max_Views")
    ) \
    .orderBy("Keyword") \
    .show(truncate=False)

+----------------+-----------+---------+----------+
|Keyword         |Media_Views|Min_Views|Max_Views |
+----------------+-----------+---------+----------+
|animals         |94723961   |23448    |1582262997|
|apple           |10746930   |1954     |425478119 |
|asmr            |1741148    |2985     |33779197  |
|bed             |53893229   |4454     |524709805 |
|biology         |4121606    |55643    |23925458  |
|business        |7236355    |2270     |208293677 |
|chess           |1323294    |601      |13844095  |
|cnn             |554240     |51269    |1889320   |
|computer science|1191959    |16115    |7004107   |
|crypto          |404608     |1599     |11805668  |
|cubes           |14735344   |10146    |168546247 |
|data science    |544772     |911      |3069097   |
|education       |2684433    |6611     |17103736  |
|finance         |694223     |1195     |9450554   |
|food            |5252406    |47430    |48018833  |
|game development|724689     |1352     |6478696   |
|gaming     

In [16]:
df_video.groupBy("Keyword") \
    .agg(
        F.min("Published At").alias("Primeiro_Published_At"),
        F.max("Published At").alias("Ultimo_Published_At")
    ) \
    .orderBy("Keyword") \
    .show(truncate=False)

+----------------+---------------------+-------------------+
|Keyword         |Primeiro_Published_At|Ultimo_Published_At|
+----------------+---------------------+-------------------+
|animals         |2009-07-03           |2022-08-24         |
|apple           |2016-11-02           |2022-08-24         |
|asmr            |2020-10-15           |2022-08-24         |
|bed             |2007-07-16           |2022-08-24         |
|biology         |2009-02-16           |2022-07-30         |
|business        |2009-10-25           |2022-08-24         |
|chess           |2019-09-12           |2022-08-24         |
|cnn             |2022-07-14           |2022-08-24         |
|computer science|2009-08-20           |2022-08-12         |
|crypto          |2022-03-11           |2022-08-24         |
|cubes           |2009-02-24           |2022-08-24         |
|data science    |2018-06-23           |2022-08-24         |
|education       |2008-07-25           |2022-08-24         |
|finance         |2012-1

In [17]:
total_titles = df_video.select("title").count()
titles_unicos = df_video.select("title").distinct().count()
diferenca = total_titles - titles_unicos

print("Total de title:", total_titles)
print("Total de title únicos:", titles_unicos)
print("Diferença:", diferenca)

if diferenca == 0:
    print("Não há diferença: todos os títulos são únicos.")
else:
    print("Há diferença: existem títulos repetidos.")

Total de title: 1869
Total de title únicos: 1854
Diferença: 15
Há diferença: existem títulos repetidos.


In [18]:
df_video.show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+---------------------+----------------------------------------------------------------+--------------------------------------+
|Title                                                                                                |Video ID   |Published At|Keyword|Likes |Comments|Views   |Interaction|Year|Month|Keyword Index|Features PCA         |Features Normal                                                 |Features                              |
+-----------------------------------------------------------------------------------------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+---------------------+----------------------------------------------------------------+--------------------------------------+
|ASMR MUKBANG DOUBLE BIG 

In [19]:
df_video_data = df_video.withColumn(
    "Published At Timestamp",
    F.to_timestamp("Published At")
)

df_video_data.select("Published At", "Published At Timestamp").show(5, truncate=False)

+------------+----------------------+
|Published At|Published At Timestamp|
+------------+----------------------+
|2020-04-18  |2020-04-18 00:00:00   |
|2022-08-22  |2022-08-22 00:00:00   |
|2022-08-24  |2022-08-24 00:00:00   |
|2022-05-30  |2022-05-30 00:00:00   |
|2017-01-02  |2017-01-02 00:00:00   |
+------------+----------------------+
only showing top 5 rows



In [20]:
df_video_data.withColumn("Year", F.year("Published At Timestamp")) \
    .groupBy("Year") \
    .count() \
    .orderBy("Year") \
    .show(truncate=False)

+----+-----+
|Year|count|
+----+-----+
|2007|2    |
|2008|1    |
|2009|9    |
|2010|6    |
|2011|4    |
|2012|12   |
|2013|6    |
|2014|10   |
|2015|15   |
|2016|34   |
|2017|47   |
|2018|57   |
|2019|86   |
|2020|158  |
|2021|229  |
|2022|1193 |
+----+-----+



In [21]:
df_video_data.withColumn("Year", F.year("Published At Timestamp")) \
    .withColumn("Month", F.month("Published At Timestamp")) \
    .groupBy("Year", "Month") \
    .count() \
    .orderBy("Year", "Month") \
    .show(truncate=False)

+----+-----+-----+
|Year|Month|count|
+----+-----+-----+
|2007|7    |1    |
|2007|12   |1    |
|2008|7    |1    |
|2009|2    |2    |
|2009|6    |2    |
|2009|7    |1    |
|2009|8    |1    |
|2009|10   |1    |
|2009|12   |2    |
|2010|3    |1    |
|2010|5    |2    |
|2010|6    |1    |
|2010|9    |1    |
|2010|10   |1    |
|2011|2    |1    |
|2011|5    |1    |
|2011|9    |1    |
|2011|10   |1    |
|2012|1    |1    |
|2012|2    |3    |
+----+-----+-----+
only showing top 20 rows



In [22]:
likes_por_ano = df_video_data \
    .withColumn("Year", F.year("Published At Timestamp")) \
    .groupBy("Keyword", "Year") \
    .agg(F.avg("Likes").alias("Media_Likes_Ano"))

janela = Window.partitionBy("Keyword") \
    .orderBy("Year") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

media_acumulativa = likes_por_ano \
    .withColumn("Media_Acumulativa_Likes", F.avg("Media_Likes_Ano").over(janela)) \
    .orderBy("Keyword", "Year")

media_acumulativa.show(truncate=False)

+-------+----+------------------+-----------------------+
|Keyword|Year|Media_Likes_Ano   |Media_Acumulativa_Likes|
+-------+----+------------------+-----------------------+
|animals|2009|1357197.0         |1357197.0              |
|animals|2010|203367.0          |780282.0               |
|animals|2013|1.1025176E7       |4195246.666666667      |
|animals|2014|3381630.0         |3991842.5              |
|animals|2019|1103713.0         |3414216.6              |
|animals|2020|769652.1111111111 |2973455.851851852      |
|animals|2021|112729.75         |2564780.6944444445     |
|animals|2022|30335.214285714286|2247975.009424603      |
|apple  |2016|4144389.0         |4144389.0              |
|apple  |2021|38261.0           |2091325.0              |
|apple  |2022|19416.6           |1400688.8666666665     |
|asmr   |2020|148120.0          |148120.0               |
|asmr   |2021|363124.3333333333 |255622.16666666666     |
|asmr   |2022|13171.31111111111 |174805.21481481483     |
|bed    |2007|

In [23]:
spark.stop()